In [42]:
import os
from typing import List, TypedDict

from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.graph import END, START,StateGraph

# Load API keys and other environment variables from .env
load_dotenv()

True

In [43]:
doc = PyPDFLoader("../pdfs/Toffoli.pdf").load()
print("No of pages in the document:", len(doc))


No of pages in the document: 19


### chunking the document into smaller pieces

In [44]:
import unicodedata

In [45]:

# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=180
)

# Split PDF into chunks
chunks = text_splitter.split_documents(doc)

print("No of chunks:", len(chunks))

# See first chunk
print(chunks[0].page_content.encode("utf-8", errors="ignore").decode("utf-8")) # for handle unicode characters
print("------------------chunk metadata:-------------------")
print(chunks[0].metadata)

No of chunks: 116
Optimal Toffoli-Depth Multi-Controlled Toffoli Decomposition in 2D Qubit Layout
Anik Basu Bhaumik, Suman Dutta, Anupam Chattopadhyay
College of Computing & Data Science, Nanyang Technological University, Singapore ∗
The multi-controlled Toffoli (MCT) gate is a key primitive in quantum arithmetic, oracle construc-
tion, and quantum cryptanalysis. Although recent work has established optimal Toffoli-depth MCT
decompositions under all-to-all qubit connectivity, their realization on near-term quantum hardware
with restricted qubit connectivity remains largely unexplored. While general-purpose quantum map-
pers can route arbitrary circuits, they do not explicitly exploit the repeated interaction patterns
inherent in MCT decompositions.
------------------chunk metadata:-------------------
{'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Anik Basu Bhaumik; Suman Dutta; Anupam Chattopadhyay', 'doi': 'https://doi.org/10

In [46]:
# -----------------------------
# 4. Google Embedding Model
# -----------------------------
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

In [48]:
# -----------------------------
# 5. Generate embedding
# -----------------------------
vectors = embeddings.embed_documents(
    [chunk.page_content for chunk in chunks]
)

print("Number of vectors:", len(vectors))
print("Vector dimension:", len(vectors[0]))

print("Embedding dimensions:", len(vectors[0]))
print("First 10 values:", vectors[0][:10])

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 2.755869494s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', '

In [ ]:


from langchain_pinecone import PineconeVectorStore

vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name="rag",
    pinecone_api_key=os.environ["PINECONE_API_KEY"]
)

print("All vectors stored in Pinecone.")

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 32.040279866s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 